Downloading and verifying the Dataset:

In [ ]:
import kagglehub
kagglehub.dataset_download('sadmanhsakib/hc18-processed-dataset')

import os
print(os.listdir("/kaggle/input/datasets/sadmanhsakib/hc18-processed-dataset/"))

Binary Mask Value Remapping:

In [ ]:
from fastai.vision.all import *
import numpy as np
from PIL import Image

class FetalMask(PILMask):
    """Map mask pixels (0/255) to class indices (0/1)."""
    @classmethod
    def create(cls, fn):
        # Open grayscale masks directly; PILImage.create would force RGB conversion.
        arr = np.array(Image.open(fn))
        if arr.ndim == 3:
            arr = arr[..., 0]
        arr = (arr > 127).astype(np.uint8)
        return cls(Image.fromarray(arr, mode="L"))

def FetalMaskBlock(codes=None):
    return TransformBlock(
        type_tfms=FetalMask.create,
        item_tfms=AddMaskCodes(codes=ifnone(codes, ['background', 'fetal_head'])),
        batch_tfms=IntToFloatTensor,
    )

Calculating the Dice + Focal Combined Loss:
Focal loss focuses on hard-to-classify pixels/boundaries and Dice Loss Directly Optimizes the target metric. Combining both is crucial for medical image segmentation. 

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

def _prepare_seg_targets(targets):
    """FastAI mask batches arrive as float tensors with shape (B, 1, H, W)."""
    if targets.ndim == 4:
        targets = targets.squeeze(1)
    return targets.long()

class DiceLoss(nn.Module):
    def __init__(self, smooth=1e-6):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        targets = _prepare_seg_targets(targets)
        probs = F.softmax(logits, dim=1)
        pred = probs[:, 1]
        target = (targets == 1).float()

        intersection = (pred * target).sum(dim=(1, 2))
        union = pred.sum(dim=(1, 2)) + target.sum(dim=(1, 2))

        dice = (2.0 * intersection + self.smooth) / (union + self.smooth)
        return 1.0 - dice.mean()

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        targets = _prepare_seg_targets(targets)
        ce_loss = F.cross_entropy(logits, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        return (self.alpha * (1.0 - pt) ** self.gamma * ce_loss).mean()

class DiceFocalLoss(nn.Module):
    """Combined Dice and Focal Loss for robust boundary segmentation."""
    def __init__(self, alpha=0.25, gamma=2.0, smooth=1e-6):
        super().__init__()
        self.dice = DiceLoss(smooth)
        self.focal = FocalLoss(alpha, gamma)

    def forward(self, logits, targets):
        return self.dice(logits, targets) + self.focal(logits, targets)

The FastAI DataBlock & Dataloaders: To ensure training matches the exact train/validation splits used for YOLOv8.

In [ ]:
def parent_folder_splitter(items):
    train_idx = [i for i, o in enumerate(items) if o.parent.name == 'train']
    val_idx = [i for i, o in enumerate(items) if o.parent.name == 'val']
    return train_idx, val_idx

def get_mask_path(img_path):
    return img_path.parent.parent / 'masks' / img_path.parent.name / img_path.name

FASTAI_DIR = Path("/kaggle/input/datasets/sadmanhsakib/hc18-processed-dataset/fastai/")

fetal_block = DataBlock(
    blocks=(ImageBlock, FetalMaskBlock(codes=['background', 'fetal_head'])),
    get_items=get_image_files,
    get_y=get_mask_path,
    splitter=parent_folder_splitter,
    item_tfms=Resize(256, method=ResizeMethod.Squish),
    batch_tfms=aug_transforms(
        mult=1.0,
        do_flip=True,
        flip_vert=False, # fetal head has anatomical up/down orientation
        max_rotate=15.0,
        max_zoom=1.15,
        max_lighting=0.15,
        max_warp=0.0  # warp is not valid for ultrasound geometry
    )
)

dls = fetal_block.dataloaders(FASTAI_DIR / "images", bs=8, num_workers=0)

# ✅ Sanity check — must print [0 1]
sample_x, sample_y = dls.train.dataset[0]
print("Mask unique values:", np.unique(np.array(sample_y)))  # expect [0 1]
print("Mask type:", type(sample_y))                          # expect FetalMask

dls.show_batch(max_n=4, vmin=0, vmax=1)

Installing Dependencies:

In [ ]:
%pip install -q onnx onnxruntime

Training the U-Net Learner:

In [ ]:
learn = unet_learner(
    dls,
    resnet34,
    loss_func=DiceFocalLoss(),
    metrics=[DiceMulti()],
)

stage_scores = []

suggested = learn.lr_find()
lr_max = suggested.valley or 3e-3
print(f"Suggested LR: {lr_max}")

learn.fit_one_cycle(
    30,
    lr_max=lr_max,
    cbs=[SaveModelCallback(monitor='dice_multi', comp=np.greater, fname='unet_frozen')],
)

frozen_loss, frozen_dice = learn.validate()
stage_scores.append(("unet_frozen", frozen_dice))
print(f"Frozen-stage validation Dice: {frozen_dice:.4f}")

learn.unfreeze()

learn.fit_one_cycle(
    20,
    lr_max=slice(1e-6, 1e-4),
    cbs=[SaveModelCallback(monitor='dice_multi', comp=np.greater, fname='unet_unfrozen')],
)

unfrozen_loss, unfrozen_dice = learn.validate()
stage_scores.append(("unet_unfrozen", unfrozen_dice))
print(f"Unfrozen-stage validation Dice: {unfrozen_dice:.4f}")

learn.fit_one_cycle(
    10,
    lr_max=slice(1e-7, 5e-5),
    cbs=[SaveModelCallback(monitor='dice_multi', comp=np.greater, fname='unet_refined')],
)

refined_loss, refined_dice = learn.validate()
stage_scores.append(("unet_refined", refined_dice))
print(f"Refined-stage validation Dice: {refined_dice:.4f}")

best_stage_name, best_stage_dice = max(stage_scores, key=lambda item: item[1])
print(f"Best stage so far: {best_stage_name} (Dice: {best_stage_dice:.4f})")

# SaveModelCallback reloads the best checkpoint for each stage.


Exporting the best FastAI U-Net to ONNX:

In [ ]:
import onnx
import onnxruntime as ort

best_checkpoint = max(stage_scores, key=lambda item: item[1])[0] if "stage_scores" in globals() and stage_scores else "unet_refined"

learn.load(best_checkpoint, with_opt=False, strict=False)
pytorch_model = learn.model.eval().cpu()

dummy_input = torch.randn(1, 3, 256, 256)
onnx_path = "/kaggle/working/unet_hc.onnx"

print(f"Exporting checkpoint '{best_checkpoint}' to {onnx_path}")

torch.onnx.export(
    pytorch_model,
    dummy_input,
    onnx_path,
    export_params=True,
    opset_version=14,
    do_constant_folding=True,
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}},
    dynamo=False,
)

onnx.checker.check_model(onnx.load(onnx_path))
print(f"ONNX graph validated. Exported to {onnx_path}")

ort_session = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
ort_session.run(None, {ort_session.get_inputs()[0].name: dummy_input.numpy()})
print("ONNX Runtime inference successful.")